In [10]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
from datetime import datetime


# =========================================================
# Configuration
# =========================================================

URL = "https://thebanks.eu/compare-banking-products/mortgage-loans/Germany"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/139.0.0.0 Safari/537.36"
    )
}

BASE_URL = "https://thebanks.eu"


# =========================================================
# Helpers
# =========================================================

def clean_text(text):
    """Normalize whitespace."""

    if not text:
        return None

    return re.sub(r"\s+", " ", text).strip()


def parse_number(text):
    """
    Convert:

        50,000.00 EUR -> 50000.0
        2.95 %        -> 2.95
        10 years      -> 10
    """

    if not text:
        return None

    match = re.search(
        r"\d+(?:,\d{3})*(?:\.\d+)?",
        text
    )

    if not match:
        return None

    return float(
        match.group().replace(",", "")
    )


def parse_range(text):
    """
    Parse:

        2.95 - 3.54 %
        50,000.00 - 999,999.00 EUR
        10 - 20 years

    Returns:

        min_value, max_value
    """

    if not text:
        return None, None

    numbers = re.findall(
        r"\d+(?:,\d{3})*(?:\.\d+)?",
        text
    )

    numbers = [
        float(x.replace(",", ""))
        for x in numbers
    ]

    if len(numbers) >= 2:
        return numbers[0], numbers[1]

    if len(numbers) == 1:
        return numbers[0], numbers[0]

    return None, None


def make_bank_id(bank_name):
    """
    Generate a stable bank_id from bank name.

    Example:
        Sparda-Bank München
        -> sparda_bank_munchen
    """

    if not bank_name:
        return None

    bank_id = bank_name.lower()

    # Replace German characters
    replacements = {
        "ä": "ae",
        "ö": "oe",
        "ü": "ue",
        "ß": "ss"
    }

    for old, new in replacements.items():
        bank_id = bank_id.replace(old, new)

    # Replace everything except letters/numbers
    bank_id = re.sub(
        r"[^a-z0-9]+",
        "_",
        bank_id
    )

    # Remove leading/trailing _
    bank_id = bank_id.strip("_")

    return bank_id


# =========================================================
# Download page
# =========================================================

response = requests.get(
    URL,
    headers=HEADERS,
    timeout=30
)

response.raise_for_status()

print("HTTP status:", response.status_code)


# =========================================================
# Parse HTML
# =========================================================

soup = BeautifulSoup(
    response.text,
    "html.parser"
)


# =========================================================
# Page-level metadata
# =========================================================

page_text = clean_text(
    soup.get_text(" ", strip=True)
)


# =========================================================
# Updated date
# =========================================================

updated_at = None

match = re.search(
    r"Updated on\s+(\d{1,2}\s+\w+\s+\d{4})",
    page_text,
    re.I
)

if match:

    updated_at = pd.to_datetime(
        match.group(1),
        dayfirst=True
    ).date()


# =========================================================
# Number of offers
# =========================================================

total_offers = None

match = re.search(
    r"(\d+)\s+offers",
    page_text,
    re.I
)

if match:

    total_offers = int(
        match.group(1)
    )


# =========================================================
# Number of banks
# =========================================================

total_banks = None

match = re.search(
    r"offers\s+from\s+(\d+)\s+banks",
    page_text,
    re.I
)

if match:

    total_banks = int(
        match.group(1)
    )


# =========================================================
# Overall effective rate
# =========================================================

overall_rate_min = None
overall_rate_max = None

match = re.search(
    r"Effective Rates?:\s*([\d.]+)\s*-\s*([\d.]+)\s*%",
    page_text,
    re.I
)

if match:

    overall_rate_min = float(
        match.group(1)
    )

    overall_rate_max = float(
        match.group(2)
    )


# =========================================================
# Overall loan term
# =========================================================

overall_term_min = None
overall_term_max = None

match = re.search(
    r"Loan Terms?:\s*(\d+)\s*-\s*(\d+)\s*years",
    page_text,
    re.I
)

if match:

    overall_term_min = int(
        match.group(1)
    )

    overall_term_max = int(
        match.group(2)
    )


# =========================================================
# Print page metadata
# =========================================================

print("\nPAGE METADATA")
print("-" * 50)

print("Updated:", updated_at)
print("Offers:", total_offers)
print("Banks:", total_banks)

print(
    "Rate:",
    overall_rate_min,
    "-",
    overall_rate_max
)

print(
    "Terms:",
    overall_term_min,
    "-",
    overall_term_max
)


# =========================================================
# Find Customer Group markers
# =========================================================

customer_groups = soup.find_all(
    string=re.compile(
        r"^\s*Customer Group\s*$",
        re.I
    )
)

print(
    "\nCustomer Group markers found:",
    len(customer_groups)
)


# =========================================================
# Bank name extraction
# =========================================================

def extract_bank_name(container):

    # -----------------------------------------------------
    # 1. Look for /banks/ links
    # -----------------------------------------------------

    for a in container.find_all("a"):

        href = a.get(
            "href",
            ""
        )

        text = clean_text(
            a.get_text(
                " ",
                strip=True
            )
        )

        if (
            "/banks/" in href
            and text
        ):
            return text


    # -----------------------------------------------------
    # 2. Look for headings
    # -----------------------------------------------------

    for tag in container.find_all(
        [
            "h1",
            "h2",
            "h3",
            "h4",
            "strong",
            "b"
        ]
    ):

        text = clean_text(
            tag.get_text(
                " ",
                strip=True
            )
        )

        if not text:
            continue

        excluded = {
            "Customer Group",
            "Amount",
            "Effective Rate",
            "Term",
            "View Details",
            "View Details +",
            "Apply for Loan",
            "Germany",
            "DE"
        }

        if text in excluded:
            continue

        # Ignore numbers
        if re.fullmatch(
            r"[\d\s.,%€EUR-]+",
            text
        ):
            continue

        if len(text) > 2:
            return text


    return None


# =========================================================
# Find container containing bank + offer
# =========================================================

def find_offer_container(marker):

    """
    Start at Customer Group and walk up the DOM.

    We want the smallest parent that contains:
        - Customer Group
        - Effective Rate
        - Term
        - bank name
    """

    parent = marker

    candidates = []

    for level in range(1, 12):

        parent = parent.parent

        if parent is None:
            break

        text = clean_text(
            parent.get_text(
                " ",
                strip=True
            )
        )

        if not text:
            continue

        # Prevent accidentally selecting the whole page
        if len(text) > 5000:
            continue

        if (
            re.search(
                r"Customer Group",
                text,
                re.I
            )
            and
            re.search(
                r"Effective Rate",
                text,
                re.I
            )
            and
            re.search(
                r"Term",
                text,
                re.I
            )
        ):

            candidates.append(parent)


    # We want the smallest valid candidate that has
    # a bank name.

    for candidate in candidates:

        bank_name = extract_bank_name(
            candidate
        )

        if bank_name:

            return candidate, bank_name


    return None, None


# =========================================================
# Parse offers
# =========================================================

offers = []

seen_offer_keys = set()


for marker in customer_groups:

    # -----------------------------------------------------
    # Find offer container
    # -----------------------------------------------------

    container, bank_name = find_offer_container(
        marker
    )

    if container is None:

        print(
            "\nWARNING: Could not find offer container"
        )

        continue


    # -----------------------------------------------------
    # Container text
    # -----------------------------------------------------

    container_text = clean_text(
        container.get_text(
            " ",
            strip=True
        )
    )


    # -----------------------------------------------------
    # Bank ID
    # -----------------------------------------------------

    bank_id = make_bank_id(
        bank_name
    )


    # =====================================================
    # Customer group
    # =====================================================

    customer_group = None

    match = re.search(
        r"Customer Group\s+([A-Z]{2})",
        container_text,
        re.I
    )

    if match:

        customer_group = (
            match.group(1).upper()
        )


    # =====================================================
    # Country
    # =====================================================

    # This scraper is specifically for Germany.
    country = "Germany"


    # =====================================================
    # Amount
    # =====================================================

    amount_min = None
    amount_max = None
    amount_raw = None

    match = re.search(
        r"Amount\s+(.+?)\s+Effective Rate",
        container_text,
        re.I
    )

    if match:

        amount_raw = clean_text(
            match.group(1)
        )

        amount_min, amount_max = parse_range(
            amount_raw
        )


    # =====================================================
    # Effective rate
    # =====================================================

    rate_min = None
    rate_max = None
    rate_raw = None

    match = re.search(
        r"Effective Rate\s+(.+?)\s+Term",
        container_text,
        re.I
    )

    if match:

        rate_raw = clean_text(
            match.group(1)
        )

        rate_min, rate_max = parse_range(
            rate_raw
        )


    # =====================================================
    # Term
    # =====================================================

    term_min = None
    term_max = None
    term_raw = None

    match = re.search(
        r"Term\s+(.+?)"
        r"\s+(?:View Details|View Details \+|Apply for Loan)",
        container_text,
        re.I
    )

    if match:

        term_raw = clean_text(
            match.group(1)
        )

        term_min, term_max = parse_range(
            term_raw
        )


    # =====================================================
    # Image
    # =====================================================

    image_url = None

    image = container.find("img")

    if image:

        image_url = image.get(
            "src"
        )

        if image_url:

            image_url = urljoin(
                BASE_URL,
                image_url
            )


    # =====================================================
    # Details URL
    # =====================================================

    details_url = None

    for a in container.find_all("a"):

        href = a.get(
            "href",
            ""
        )

        if href:

            # Mortgage detail links
            if (
                "#" in href
                and
                "mortgage-loans" in href
            ):

                details_url = urljoin(
                    BASE_URL,
                    href
                )

                break


    # =====================================================
    # Duplicate protection
    # =====================================================

    offer_key = (
        bank_id,
        amount_raw,
        rate_raw,
        term_raw
    )

    if offer_key in seen_offer_keys:

        continue

    seen_offer_keys.add(
        offer_key
    )


    # =====================================================
    # Save offer
    # =====================================================

    offers.append({

        # -------------------------------------------------
        # Page metadata
        # -------------------------------------------------

        "updated_at": updated_at,

        "total_offers": total_offers,

        "total_banks": total_banks,

        "overall_rate_min_pct":
            overall_rate_min,

        "overall_rate_max_pct":
            overall_rate_max,

        "overall_term_min_years":
            overall_term_min,

        "overall_term_max_years":
            overall_term_max,


        # -------------------------------------------------
        # Bank
        # -------------------------------------------------

        "bank_id": bank_id,

        "bank_name": bank_name,

        "country": country,

        "customer_group": customer_group,


        # -------------------------------------------------
        # Amount
        # -------------------------------------------------

        "amount_min_eur": amount_min,

        "amount_max_eur": amount_max,

        "amount_raw": amount_raw,


        # -------------------------------------------------
        # Effective rate
        # -------------------------------------------------

        "effective_rate_min_pct":
            rate_min,

        "effective_rate_max_pct":
            rate_max,

        "effective_rate_raw":
            rate_raw,


        # -------------------------------------------------
        # Term
        # -------------------------------------------------

        "term_min_years":
            term_min,

        "term_max_years":
            term_max,

        "term_raw":
            term_raw,


        # -------------------------------------------------
        # URLs
        # -------------------------------------------------

        "details_url": details_url,

        "image_url": image_url,


        # -------------------------------------------------
        # Technical
        # -------------------------------------------------

        "scraped_at": datetime.utcnow()
    })


# =========================================================
# Create DataFrame
# =========================================================

df = pd.DataFrame(
    offers
)


# =========================================================
# Sort
# =========================================================

if not df.empty:

    df = df.sort_values(
        [
            "bank_name",
            "effective_rate_min_pct"
        ],
        na_position="last"
    ).reset_index(
        drop=True
    )


# =========================================================
# Final output
# =========================================================

print("\n")
print("=" * 80)
print("SCRAPING RESULT")
print("=" * 80)

print(
    "Rows parsed:",
    len(df)
)

print(
    "Unique banks:",
    df["bank_name"].nunique()
    if not df.empty
    else 0
)


# =========================================================
# Show important columns
# =========================================================

if not df.empty:

    print("\nDATA:")
    print("-" * 80)

    print(
        df[
            [
                "bank_id",
                "bank_name",
                "country",
                "amount_raw",
                "effective_rate_raw",
                "term_raw"
            ]
        ].to_string(
            index=False
        )
    )

else:

    print(
        "\nWARNING: DataFrame is empty."
    )

    print(
        "\nThe page structure probably changed."
    )


# =========================================================
# Optional: save CSV
# =========================================================

df.to_csv(
    "germany_mortgage_offers.csv",
    index=False
)

print(
    "\nSaved to: germany_mortgage_offers.csv"
)

HTTP status: 200

PAGE METADATA
--------------------------------------------------
Updated: 2026-09-16
Offers: 13
Banks: 7
Rate: 2.95 - 4.92
Terms: 5 - 30

Customer Group markers found: 13


SCRAPING RESULT
Rows parsed: 13
Unique banks: 8

DATA:
--------------------------------------------------------------------------------
                                  bank_id                                  bank_name country                                                                    amount_raw effective_rate_raw      term_raw
bnp_paribas_s_a_niederlassung_deutschland BNP Paribas S.A. Niederlassung Deutschland Germany                                                  50,000.00 - 1,000,000.00 EUR      3.66 - 4.92 %          None
                    frankfurter_sparkasse                      Frankfurter Sparkasse Germany                                                                          None      3.81 - 4.01 %          None
                              ing_diba_ag                    

/var/folders/fy/v_csgbfs5292myngmd5dzk4m0000gn/T/ipykernel_35366/665877652.py:804: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at": datetime.utcnow()
